# Inventory EDA — Youth Services + ACS Baselines (San Diego)
_Quick structure checks, missingness, simple counts, and standardized outputs_

This notebook:
- Loads the tract-level ACS baseline.
- Loads raw services CSVs (YMCA, public libraries, City rec centers, parks).
- Normalizes columns to a common schema: ['source','name','address','city','state','zip','lat','lon','raw_id'].
- Does light EDA: .info(), head(), missingness, duplicates, basic counts.
- Loads GTFS stops to sanity-check transit coverage (count & sample).
- Writes cleaned CSVs to `data/interim/services_clean/`.

In [1]:
from __future__ import annotations
import pandas as pd
import numpy as np
from pathlib import Path
import re

pd.set_option("display.max_columns", 150)
pd.set_option("display.width", 160)

In [ ]:
# Repo-root detection
def find_repo_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    markers = ("data", ".git", "pyproject.toml", "Makefile")
    while p.parent != p and not any((p / m).exists() for m in markers):
        p = p.parent
    return p

REPO = find_repo_root()
DATA = REPO / "data"
RAW  = DATA / "raw"
INTERIM = DATA / "interim"
PROCESSED = DATA / "processed"
INTERIM.mkdir(parents=True, exist_ok=True)

print("Repo root:", REPO)

# Paths
ACS_BASE = PROCESSED / "acs_baselines_tract_2023_clean.csv"  # created in previous notebook

SERV_RAW = RAW / "services"
SERV_OUT = INTERIM / "services_clean"
SERV_OUT.mkdir(parents=True, exist_ok=True)

YMCA_F    = SERV_RAW / "ymca_sd.csv"
LIBS_F    = SERV_RAW / "sd_public_libraries.csv"   # county libs
LIBS2_F   = SERV_RAW / "Library.csv"               # if you want to compare/replace later
REC_F     = SERV_RAW / "rec_centers_datasd.csv"    # alt 1
REC2_F    = SERV_RAW / "Recreation_Centers_SD.csv" # alt 2
PARKS_F   = SERV_RAW / "Parks_SD.csv"

GTFS_STOPS_F  = SERV_RAW / "Transit_Stops_GTFS_20251028.csv"
GTFS_ROUTES_F = SERV_RAW / "Transit_Routes_GTFS.csv"

for p in [ACS_BASE, YMCA_F, LIBS_F, REC_F, REC2_F, PARKS_F, GTFS_STOPS_F, GTFS_ROUTES_F, LIBS2_F]:
    print(f"{p.relative_to(REPO)!s:<80} exists={p.exists()}")

Repo root: /Users/laurenvo/Documents/github/Mapping-Youth-Opportunity-Deserts
data/processed/acs_baselines_tract_2023_clean.csv                                exists=True
data/raw/services/ymca_sd.csv                                                    exists=True
data/raw/services/sd_public_libraries.csv                                        exists=True
data/raw/services/rec_centers_datasd.csv                                         exists=True
data/raw/services/Recreation_Centers_SD.csv                                      exists=True
data/raw/services/Parks_SD.csv                                                   exists=True
data/raw/services/Transit_Stops_GTFS_20251028.csv                                exists=True
data/raw/services/Transit_Routes_GTFS.csv                                        exists=True
data/raw/services/Library.csv                                                    exists=True


In [7]:
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Iterable

def snake(s: str) -> str:
    return (
        s.strip()
         .replace("/", "_")
         .replace("-", "_")
         .replace("(", "")
         .replace(")", "")
         .replace(" ", "_")
         .lower()
    )

def read_csv_clean(path: Path, **kwargs) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False, **kwargs)
    df.columns = [snake(c) for c in df.columns]
    return df

def coerce_numeric(df: pd.DataFrame, cols: Iterable[str]) -> pd.DataFrame:
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def unify_lat_lon(df: pd.DataFrame) -> pd.DataFrame:
    # Try common lat/lon header variants and consolidate to 'lat' / 'lon'
    lat_candidates = [c for c in df.columns if c in {"lat","latitude","y","geom_y","stop_lat"}]
    lon_candidates = [c for c in df.columns if c in {"lon","lng","longitude","x","geom_x","stop_lon"}]

    if "lat" not in df.columns:
        for c in lat_candidates:
            df = df.rename(columns={c: "lat"})
            break
    if "lon" not in df.columns and "lng" in df.columns:
        df = df.rename(columns={"lng": "lon"})
    if "lon" not in df.columns:
        for c in lon_candidates:
            df = df.rename(columns={c: "lon"})
            break

    # Coerce to numeric
    df = coerce_numeric(df, ["lat", "lon"])
    return df

def quick_point_checks(name: str, df: pd.DataFrame) -> None:
    n = len(df)
    miss_lat = df["lat"].isna().sum() if "lat" in df.columns else n
    miss_lon = df["lon"].isna().sum() if "lon" in df.columns else n
    dup_name_addr = 0
    for a in ("address","addr","site_address","street_address"):
        if a in df.columns:
            k = ["name", a] if "name" in df.columns else [a]
            dup_name_addr = df.duplicated(subset=[c for c in k if c in df.columns]).sum()
            break

    print(f"{name:20s} rows={n:5d}  miss_lat={miss_lat:4d}  miss_lon={miss_lon:4d}  duplicates~name+addr={dup_name_addr}")


In [ ]:
# ACS baseline (tract level)
acs = read_csv_clean(ACS_BASE, dtype={"GEOID": "string"})
# Keep expected columns only if present
acs_expected = [
    "geoid","name","pop_total","youth_5_17","youth_10_19",
    "youth_5_17_per_1k","youth_10_19_per_1k",
    "median_hh_income_2023usd","households_total","households_zero_veh","zero_veh_share"
]
acs = acs[[c for c in acs_expected if c in acs.columns]].copy()

print("ACS rows:", len(acs))
display(acs.head(3))
display(acs.select_dtypes(include="number").describe().T)

# YMCA
ymca = unify_lat_lon(read_csv_clean(YMCA_F))
# Libraries (county list)
libs = unify_lat_lon(read_csv_clean(LIBS_F))
# Libraries (alternate / state list)
libs2 = unify_lat_lon(read_csv_clean(LIBS2_F))
# Rec centers (dataset A)
rec_a = unify_lat_lon(read_csv_clean(REC_F))
# Rec centers (dataset B)
rec_b = unify_lat_lon(read_csv_clean(REC2_F))
# Parks
parks = unify_lat_lon(read_csv_clean(PARKS_F))
# GTFS stops and routes
gtfs_stops  = unify_lat_lon(read_csv_clean(GTFS_STOPS_F))
gtfs_routes = read_csv_clean(GTFS_ROUTES_F)  # no lat/lon expected here

ACS rows: 737


,geoid,name,pop_total,youth_5_17,youth_10_19,youth_5_17_per_1k,youth_10_19_per_1k,median_hh_income_2023usd,households_total,households_zero_veh,zero_veh_share
0,06073000100,Census Tract 1; San Diego County; California,2746.0,360.0,310.0,131.099782,112.891479,211023.0,1103.0,10.0,0.009066
1,06073000201,Census Tract 2.01; San Diego County; California,2376.0,355.0,279.0,149.410774,117.424242,110476.0,1258.0,155.0,0.123211
2,06073000202,Census Tract 2.02; San Diego County; California,4019.0,344.0,247.0,85.593431,61.458074,121264.0,2253.0,90.0,0.039947


,count,mean,std,min,25%,50%,75%,max
pop_total,737.0,4454.249661,2107.362836,0.0,3313.000000,4257.000000,5436.000000,40481.000000
youth_5_17,737.0,687.552239,437.399949,0.0,422.000000,633.000000,897.000000,4064.000000
youth_10_19,737.0,553.706920,426.976540,0.0,316.000000,494.000000,703.000000,5984.000000
youth_5_17_per_1k,735.0,148.527286,60.066114,0.0,116.088617,155.036095,187.048111,338.547747
youth_10_19_per_1k,735.0,119.281729,59.520336,0.0,85.907905,117.724519,146.556075,688.869153
median_hh_income_2023usd,724.0,109939.169890,42330.828767,24125.0,78217.250000,104273.500000,133967.000000,246078.000000
households_total,737.0,1573.706920,621.646224,0.0,1167.000000,1523.000000,1923.000000,6992.000000
households_zero_veh,737.0,85.563094,96.705004,0.0,21.000000,55.000000,119.000000,963.000000
zero_veh_share,732.0,0.055036,0.065431,0.0,0.016225,0.036865,0.075102,1.000000


In [ ]:
for df in (ymca, libs, libs2, rec_a, rec_b, parks):
    # Normalize common descriptive fields if present
    for src, dst in {
        "org": "organization",
        "facility": "name",
        "park_name": "name",
        "rec_bldg": "name",
        "stop_name": "name",
    }.items():
        if src in df.columns and dst not in df.columns:
            df.rename(columns={src: dst}, inplace=True)

    # Basic trimming
    for c in ("name","organization","address","city","state","zip"):
        if c in df.columns and df[c].dtype == "object":
            df[c] = df[c].astype(str).str.strip()


In [ ]:
def keep_valid_points(df: pd.DataFrame) -> pd.DataFrame:
    df2 = df.copy()
    if "lat" in df2.columns and "lon" in df2.columns:
        df2 = df2.dropna(subset=["lat","lon"])
    # remove obvious zero points
    if "lat" in df2.columns and "lon" in df2.columns:
        df2 = df2[(df2["lat"].between(-90,90)) & (df2["lon"].between(-180,180))]
    # light dedup by name+address if available
    keys = [k for k in ("name","address") if k in df2.columns]
    if keys:
        df2 = df2.drop_duplicates(subset=keys, keep="first")
    return df2

ymca      = keep_valid_points(ymca)
libs      = keep_valid_points(libs)
libs2     = keep_valid_points(libs2)
rec_a     = keep_valid_points(rec_a)
rec_b     = keep_valid_points(rec_b)
parks     = keep_valid_points(parks)
gtfs_stops= keep_valid_points(gtfs_stops)

In [ ]:
quick_point_checks("YMCA", ymca)
quick_point_checks("Libraries A", libs)
quick_point_checks("Libraries B", libs2)
quick_point_checks("Rec centers A", rec_a)
quick_point_checks("Rec centers B", rec_b)
quick_point_checks("Parks", parks)
quick_point_checks("GTFS stops", gtfs_stops)

print("\nGTFS routes columns:", list(gtfs_routes.columns)[:12], "…")
print("GTFS routes rows:", len(gtfs_routes))

for name, df in {
    "YMCA": ymca, "Libraries A": libs, "Libraries B": libs2,
    "Rec centers A": rec_a, "Rec centers B": rec_b, "Parks": parks,
    "GTFS stops": gtfs_stops
}.items():
    print(f"\n=== {name} sample ===")
    display(df.head(3))

YMCA                 rows=   23  miss_lat=  23  miss_lon=  23  duplicates~name+addr=0
Libraries A          rows=   38  miss_lat=  38  miss_lon=  38  duplicates~name+addr=0
Libraries B          rows=   92  miss_lat=  92  miss_lon=  92  duplicates~name+addr=0
Rec centers A        rows=   63  miss_lat=   0  miss_lon=   0  duplicates~name+addr=0
Rec centers B        rows=    0  miss_lat=   0  miss_lon=   0  duplicates~name+addr=0
Parks                rows=  301  miss_lat= 301  miss_lon= 301  duplicates~name+addr=0
GTFS stops           rows= 6230  miss_lat=   0  miss_lon=   0  duplicates~name+addr=0

GTFS routes columns: ['oid_', 'shape_id', 'route_id', 'route_short_name', 'route_long_name', 'route_type', 'agency_id', 'route_desc', 'route_url', 'route_color', 'route_text_color', 'route_type_text'] …
GTFS routes rows: 478

=== YMCA sample ===


,organization,name,address,city,state,zip,phone,fax,website,source
0,YMCA of San Diego County,Border View Family YMCA,3601 Arey Drive,San Diego,CA,91154,(619) 428-9622,(619) 298-9262,https://www.ymcasd.org/locations/border-view-f...,https://www.ymcasd.org/locations/
1,YMCA of San Diego County,Cameron Family YMCA,10123 Riverwalk Drive,Santee,CA,92071,(619) 449-9622,(619) 449-9624,https://www.ymcasd.org/locations/cameron-famil...,https://www.ymcasd.org/locations/
2,YMCA of San Diego County,Copley-Price Family YMCA,4300 El Cajon Boulevard,San Diego,CA,92105,(619) 280-9622,(619) 283-7586,https://www.ymcasd.org/locations/copley-price-...,https://www.ymcasd.org/locations/



=== Libraries A sample ===


,organization,name,address,city,state,zip,phone,website,source
0,San Diego Public Library,Central Library,330 Park Blvd,San Diego,CA,92101,(619) 236-5800,https://www.sandiego.gov/public-library/centra...,https://sandiego.librarymarket.com/branches/se...
1,San Diego Public Library,Allied Gardens/Benjamin Library,5188 Zion Ave,San Diego,CA,92120,(619) 533-3970,NaN,https://sandiego.librarymarket.com/branches/se...
2,San Diego Public Library,Balboa Library,4255 Mt. Abernathy Ave,San Diego,CA,92117,(858) 573-1390,NaN,https://sandiego.librarymarket.com/branches/se...



=== Libraries B sample ===


...

,objectid,rec_bldg,name,address,city,zipcode,square_feet,year_built,adult_center,comfort_station,comp_room,dance_room,game_room,gymnasium,kiln,kiln_room,kitchen,multi_purpose,racquetball,stage,teen_center,tiny_tot_room,weight_room,neighborhood,cd,service_district,indoor_bball,indoor_vball,indoor_pickleball,notes,maint_entity,community,facility_name_id,globalid,lon,lat



=== Parks sample ===


,objectid,common_name,desig_use,acres,address_lo,community,use_restriction,dedicated_park,full_name,undeveloped_ac,useable_ac,gross_ac,maint_entity,recycled_water,play_area_cnt,cd,dms,x_coord,y_coord,year_acquired,read_ac,notes,year_constructed,year_assessed,pci,service_district,playground,tot_lot,baseball_50_6,baseball_90,softball,multi_purpose,f_2_baselet,basketball,tennis,sand_vball,comfort_station,concession_stand,pickleball,field_lighting_cnt,field_lighting,created_user,created_date,last_edited_user,last_edited_date,globalid,crg,gdp_url,fitness_st,shape_length,shape_area
0,1,OAK PARK NP,Neighborhood Park,3.488897,"5235 Maple St. , 92105",MID-CITY: EASTERN AREA,DEDIC PK/OS--ORD.,Y,Oak Neighborhood Park,0.0,0.0,0.0,CP2,N,1.0,4,32 43 57.17844N 117 04 58.82675W,-117.083007,32.732550,1949,3.460,NaN,1974,2018.0,15,41.0,1,0,0,0,1,0,0,0,0,0,0,0,0,0.0,N,NaN,NaN,RSAIGA,11/6/2023 23:37:14,{778A7D4D-2CFB-42A5-B8C7-FC2B67CD579F},Colina Del Sol,https://www.sandiego.gov/sites/default/files/p...,0,1560.601140,151975.743542
1,2,SAN CARLOS CP & REC CENTER,Community Park,9.788830,"6445 Lake Badin Ave., 92119",NAVAJO,DEDIC PK/OS--ORD.,Y,San Carlos Community Park & Recreation Center,0.0,0.0,0.0,CP1,N,1.0,7,32 47 55.12525N 117 01 16.13407W,-117.021148,32.798646,1961-1976,10.153,NaN,1967,2015.0,10,45.0,1,1,0,0,2,1,0,2,0,0,0,0,0,0.0,N,NaN,NaN,RSAIGA,11/6/2023 23:37:14,{7F9F8E68-130C-4201-8D72-D6834E38A633},San Carlos,https://www.sandiego.gov/sites/default/files/p...,0,2651.794044,426399.726939
2,3,MARCY NP,Neighborhood Park,10.887424,"5526 STRESEMANN ST, 92122",UNIVERSITY,VOTER RATIF. REQ'D,Y,Marcy Neighborhood Park,0.0,0.0,0.0,CP1,N,1.0,6,32 50 42.50195N 117 13 37.81278W,-117.227170,32.845139,1961,10.960,NaN,1964,2015.0,29,44.0,1,0,0,0,0,0,0,0,0,0,0,0,0,0.0,N,NaN,NaN,RSAIGA,11/6/2023 23:37:14,{C74D60C3-B4BF-431D-8D1B-A41EAA729F05},Standley,https://www.sandiego.gov/sites/default/files/p...,0,3709.224462,474254.282819



=== GTFS stops sample ===


,the_geom,objectid,stop_uid,stop_agency,stop_id,stop_name,lat,lon,stop_code,location_type,parent_station,wheelchair_boarding,intersection_code,stop_place
0,POINT (-117.225253999814 33.289260000306),"5,239",NCTD_21699,NCTD,21699,Hwy 76 & Camino Del Rey,33.289260,-117.225254,21699.0,0,NaN,NaN,F-N/B,miscam
1,POINT (-117.090831370467 32.658396440078),"1,947",MTS_12532,MTS,12532,Highland Av & 30th St,32.658396,-117.090831,12532.0,0,NaN,1.0,N-N/B,high30
2,POINT (-117.146401100416 32.753688689998),"3,404",MTS_13553,MTS,13553,Park Bl & Howard Av,32.753689,-117.146401,13553.0,0,NaN,0.0,F-S/B,NaN


In [ ]:
INTERIM.mkdir(parents=True, exist_ok=True)

ymca.to_csv(  INTERIM / "ymca_points_clean.csv", index=False)
libs.to_csv(   INTERIM / "libraries_points_clean.csv", index=False)
libs2.to_csv(  INTERIM / "libraries_points_clean_alt.csv", index=False)
rec_a.to_csv(  INTERIM / "rec_centers_A_points_clean.csv", index=False)
rec_b.to_csv(  INTERIM / "rec_centers_B_points_clean.csv", index=False)
parks.to_csv(  INTERIM / "parks_points_clean.csv", index=False)
gtfs_stops.to_csv(INTERIM / "gtfs_stops_points_clean.csv", index=False)

print("\nWrote interim cleaned CSVs to:", INTERIM)


Wrote interim cleaned CSVs to: /Users/laurenvo/Documents/github/Mapping-Youth-Opportunity-Deserts/data/interim
